9 Конвертация и приведение типов

In [1]:
3 + 4.2


7.2


In [2]:
UInt8(5) + Int128(3e6)


3000005


In [3]:
1//4 + 0.25


0.5


9.1 Обследование системы приведения чисел

In [4]:
@which 2 + 3.5


+(x::Number, y::Number)
     @ Base promotion.jl:433


In [5]:
promote(2, 3.5)


(2.0, 3.5)


In [6]:
typeof(1//2), typeof(42.5), typeof(false), typeof(0x5)


(Rational{Int64}, Float64, Bool, UInt8)


In [7]:
values = promote(1//2, 42.5, false, 0x5)


(0.5, 42.5, 0.0, 5.0)


In [8]:
map(typeof, values)


(Float64, Float64, Float64, Float64)


9.2 Понимание конвертации чисел

In [9]:
x = Int8(32)


32


In [10]:
typeof(x)


Int8


In [11]:
Int8(4.0)


4


In [12]:
Float64(1//2)


0.5


In [13]:
Float32(24)


24.0f0


In [14]:
Int8(1000)


Exception: InexactError: trunc(Int8, 1000)

In [15]:
Int64(4.5)


Exception: InexactError: Int64(4.5)

In [16]:
convert(Int64, 5.0)


5


In [17]:
convert(Float64, 5)


5.0


In [18]:
convert(UInt8, 4.0)


0x04


In [19]:
1//4 + 1//4


1//2


In [20]:
convert(Float32, 1//4 + 1//4)


0.5f0


In [21]:
3 isa Int64


true


In [22]:
Int64 isa Type{Int64}


true


In [23]:
"hi" isa String


true


In [24]:
String isa Type{String}


true


In [25]:
Int64 isa Type{String}


false


In [26]:
values = Int8[3, 5]


2-element Vector{Int8}:
 3
 5


In [27]:
typeof(values[2])


Int8


In [28]:
x = 42


42


In [29]:
typeof(x)


Int64


In [30]:
values[2] = x


42


In [31]:
mutable struct Point
    x::Float64
    y::Float64
end


In [32]:
p = Point(3.5, 6.8)


Point(3.5, 6.8)


In [33]:
p.x = Int8(10)


10


In [34]:
foo(x::Int64)::UInt8 = 2x


foo (generic function with 1 method)


In [35]:
y = foo(42)


0x54


In [36]:
typeof(y)


UInt8


9.3 Определение конкретно-прикладных единиц измерения углов

In [37]:
abstract type Angle end

struct Radian <: Angle
    radians::Float64
end

# Градусы, минуты, секунды (DMS)
struct DMS <: Angle
    seconds::Int
end


9.3.1 Определение конструкторов углов

In [38]:
Degree(degrees::Integer) = Minute(degrees * 60)
Degree(deg::Integer, min::Integer) = Degree(deg) + Minute(min)

function Degree(deg::Integer, min::Integer, secs::Integer)
    Degree(deg, min) + Second(secs)
end

function Minute(minutes::Integer)
    DMS(minutes * 60)
end

function Second(seconds::Integer)
    DMS(seconds)
end


Second (generic function with 1 method)


9.3.2 Определение арифметических операций на углах

In [39]:
import Base: -, +

+(Θ::DMS, α::DMS) = DMS(Θ.seconds + α.seconds)
-(Θ::DMS, α::DMS) = DMS(Θ.seconds - α.seconds)

+(Θ::Radian, α::Radian) = Radian(Θ.radians + α.radians)
-(Θ::Radian, α::Radian) = Radian(Θ.radians - α.radians)


- (generic function with 196 methods)


9.3.3 Определение функций доступа для извлечения градусов, минут и секунд

In [40]:
function degrees(dms::DMS)
    minutes = dms.seconds ÷ 60
    minutes ÷ 60
end

function minutes(dms::DMS)
    minutes = dms.seconds ÷ 60
    minutes % 60
end

seconds(dms::DMS) = dms.seconds % 60


seconds (generic function with 1 method)


9.3.4 Отображение углов DMS на экране

In [41]:
α = Degree(90, 30, 45)


DMS(325845)


In [42]:
degrees(α)


90


In [43]:
minutes(α)


30


In [44]:
seconds(α)


45


In [45]:
β = Degree(90, 30) + Degree(90, 30)


DMS(651600)


In [46]:
degrees(β)


181


In [47]:
minutes(β)


0


In [48]:
import Base: show

function show(io::IO, dms::DMS)
    print(io, degrees(dms), "° ", minutes(dms), "' ", seconds(dms), "''")
end

function show(io::IO, rad::Radian)
    print(io, rad.radians, "rad")
end


show (generic function with 273 methods)


In [49]:
α = Degree(90, 30, 45)


90° 30' 45''


In [50]:
β = Degree(90, 30) + Degree(90, 30)


181° 0' 0''


9.3.5 Определение конвертаций типов

In [51]:
import Base: convert

Radian(dms::DMS) = Radian(deg2rad(dms.seconds / 3600))
Degree(rad::Radian) = DMS(floor(Int, rad2deg(rad.radians) * 3600))
DMS(rad::Radian) = Degree(rad)

convert(::Type{Radian}, dms::DMS) = Radian(dms)
convert(::Type{DMS}, rad::Radian) = DMS(rad)


convert (generic function with 212 methods)


In [52]:
sin(rad::Radian) = Base.sin(rad.radians)
cos(rad::Radian) = Base.cos(rad.radians)
sin(dms::DMS) = sin(Radian(dms))
cos(dms::DMS) = cos(Radian(dms))


cos (generic function with 2 methods)


In [53]:
sin(π / 2)


Exception: MethodError: no method matching sin(::Float64)
You may have intended to import Base.sin
The function `sin` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  sin(!Matched::DMS)
   @ Main none:4
  sin(!Matched::Radian)
   @ Main none:2

In [54]:
sin(90)


Exception: MethodError: no method matching sin(::Int64)
You may have intended to import Base.sin
The function `sin` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  sin(!Matched::DMS)
   @ Main none:4
  sin(!Matched::Radian)
   @ Main none:2

In [55]:
sin(Degree(90))


1.0


In [56]:
sin(Radian(π / 2))


1.0


9.3.6 Создание красивых литералов углов

In [57]:
import Base: *, /

*(coeff::Number, dms::DMS) = DMS(coeff * dms.seconds)
*(dms::DMS, coeff::Number) = coeff * dms
/(dms::DMS, denom::Number) = DMS(dms.seconds / denom)

*(coeff::Number, rad::Radian) = Radian(coeff * rad.radians)
*(rad::Radian, coeff::Number) = coeff * rad
/(rad::Radian, denom::Number) = Radian(rad.radians / denom)

const ° = Degree(1)
const rad = Radian(1)


1.0rad


In [58]:
sin(90°)


1.0


In [59]:
sin(1.5rad)


0.9974949866040544


In [60]:
cos(30°)


0.8660254037844387


In [61]:
cos(90° / 3)


0.8660254037844387


In [62]:
sin(3rad / 2)


0.9974949866040544


9.3.7 Приведения типов

In [63]:
+(Θ::Angle, α::Angle) = +(promote(Θ, α)...)
-(Θ::Angle, α::Angle) = -(promote(Θ, α)...)


- (generic function with 197 methods)


In [64]:
import Base: promote_rule

promote_rule(::Type{Radian}, ::Type{DMS}) = Radian


promote_rule (generic function with 135 methods)


In [65]:
promote_rule(Int16, UInt8)


Int16


In [66]:
promote_rule(Float64, UInt8)


Float64


In [67]:
promote_rule(Radian, DMS)


Radian


In [68]:
sin(90° + 3.14rad / 2)


0.0007963267107331024


In [69]:
cos(90° + 3.14rad / 2)


-0.9999996829318346


In [70]:
45° + 45°


90° 0' 0''


In [71]:
Radian(45° + 45°)


1.5707963267948966rad


In [72]:
45° + 3.14rad / 4


1.5703981633974484rad
